# TECNICATURA SUPERIOR EN DESARROLLO DE SOFTWARE
## Espacio Curricular: Desarrollo de Inteligencia Artificial
### Evidencia de Aprendizaje N.º 2: Modelado y Evaluación de Desempeño
**Profesora:** Cra. Sol Figueroa  
**Integrantes:**
* Vargas, Cristian
* Quinteros, Karina
* Zárate, Laura

---

## 1. Contexto y Planteo del Problema

En la etapa previa de **Análisis Exploratorio de Datos (Evidencia N.º 1)**, se analizaron las fuentes de datos de stock, ventas y proveedores de neumáticos. Dicho análisis estableció conclusiones fundamentales:
1. Las dimensiones físicas del neumático están codificadas en el nombre del producto (`Product Name`).
2. El diámetro del rin (`Rim`) demostró ser el mejor predictor lineal del costo unitario ($r = 0.76$), seguido por el ancho de sección (`Width`, $r = 0.44$).
3. La marca fabricante (`Brand`) exhibe diferencias estructurales de cotización (marcas de entrada masiva vs. marcas de alta gama/pesadas).
4. El EDA concluyó formalmente que esta correlación representa una **"buena señal para regresión"**.

### Preguntas Guía del Modelado:
* **¿Qué variable queremos predecir?** El costo unitario de catálogo: `Unit Cost (INR)`.
* **¿Qué variables nos ayudan a explicarla?** Las dimensiones físicas (`Rim`, `Width`, `Aspect Ratio`) y el fabricante (`Brand`).


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Configuración visual
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')


## 2. Carga del Dataset y Depuración del Target

Utilizamos el dataset de inventario estático `DS_Data_Set_Current_Stock.csv` (253 registros), donde cada fila representa un producto de catálogo individual.
Para el entrenamiento supervisado, se limpian las máscaras textuales de moneda y se descartan de forma estricta las **2 observaciones sin target** (registros con `Unit Cost` nulo). No se imputa la variable dependiente para preservar la validez empírica del estudio.


In [ ]:
# 1. Cargar datos
df_raw = pd.read_csv('../data/DS_Data_Set_Current_Stock.csv')
print(f"Dimensiones iniciales del dataset: {df_raw.shape[0]} filas, {df_raw.shape[1]} columnas")

# 2. Limpieza de la variable objetivo (Unit Cost en INR)
df_raw['unit_cost_clean'] = (
    df_raw['Unit Cost (INR)']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('"', '', regex=False)
    .str.strip()
    .pipe(pd.to_numeric, errors='coerce')
)

# 3. Eliminar filas sin costo cargado
nulos_costo = df_raw['unit_cost_clean'].isna().sum()
df = df_raw.dropna(subset=['unit_cost_clean']).copy()

print(f"Registros con costo faltante eliminados: {nulos_costo}")
print(f"Población supervisada disponible para modelado: {len(df)} observaciones")
df[['Product ID', 'Brand', 'Product Name', 'unit_cost_clean']].head(5)


## 3. Extracción de Especificaciones Físicas (Feature Engineering)

A partir de la cadena de texto `Product Name`, extraemos las variables cuantitativas físicas mediante una jerarquía taxonómica de expresiones regulares que cubre el 100% del catálogo:
1. **Nivel 1 (Ratio Explícito):** En neumáticos métricos estándar (ej. `145/70 R 12`, `225/40 Z R 19`), se extrae directamente el ancho en mm, el ratio explícito en % y el rin en pulgadas.
2. **Nivel 2 (Flotación):** En nomenclaturas norteamericanas en pulgadas (ej. `31 x 10.50 R 15`), se realiza la conversión métrica ($W = 10.50 \times 25.4 = 266.7\text{ mm}$) y se deduce geométricamente el perfil exacto a partir del diámetro exterior:
   $$H = \frac{OD - Rim}{2} = \frac{31 - 15}{2} = 8.0\text{ in} \implies \text{Ratio} = \left( \frac{8.0}{10.50} \right) \times 100 \approx 76.19\%$$
3. **Nivel 3 (Nomenclaturas sin ratio determinable):** Comerciales ligeros tipo C (`155 R 12 C`) y transporte pesado (`1000 R 20`, `11 R 22.5`). El parser no inventa valores y asigna de forma estricta `NaN` para ser tratado posteriormente en la etapa de imputación sin fuga de datos.


In [ ]:
def parse_tire_specs(name):
    s = str(name).strip()

    # Nivel 2: Flotación (ej. 31 x 10.50 R 15 LT / 6PR)
    m_flot = re.search(r'^(\d+(?:\.\d+)?)\s*x\s*(\d+(?:\.\d+)?)\s*(?:Z\s*)?R\s*(\d+(?:\.\d+)?)', s, re.I)
    if m_flot:
        od_in = float(m_flot.group(1))
        w_in = float(m_flot.group(2))
        rim = float(m_flot.group(3))
        width_mm = w_in * 25.4
        ratio = ((od_in - rim) / (2.0 * w_in)) * 100.0
        return width_mm, ratio, rim

    # Nivel 1: Métrico convencional o con código Z (ej. 145/70 R 12, 225/40 Z R 19)
    m_met = re.search(r'^(\d{3})/(\d{2})\s*(?:Z\s*)?R\s*(\d+(?:\.\d+)?)', s, re.I)
    if m_met:
        width_mm = float(m_met.group(1))
        ratio = float(m_met.group(2))
        rim = float(m_met.group(3))
        return width_mm, ratio, rim

    # Nivel 3A: Comercial ligero sin perfil explícito (ej. 155 R 12 C)
    m_com = re.search(r'^(\d{3})\s*(?:Z\s*)?R\s*(\d+(?:\.\d+)?)', s, re.I)
    if m_com:
        width_mm = float(m_com.group(1))
        rim = float(m_com.group(2))
        return width_mm, np.nan, rim

    # Nivel 3B: Camión en centésimas de pulgada (ej. 1000 R 20 -> 10.00 pulgadas = 254 mm)
    m_cam1 = re.search(r'^(\d{4})\s*(?:Z\s*)?R\s*(\d+(?:\.\d+)?)', s, re.I)
    if m_cam1:
        w_in = float(m_cam1.group(1)) / 100.0
        width_mm = w_in * 25.4
        rim = float(m_cam1.group(2))
        return width_mm, np.nan, rim

    # Nivel 3C: Camión en pulgadas enteras (ej. 11 R 22.5 -> 11.00 pulgadas = 279.4 mm)
    m_cam2 = re.search(r'^(\d{1,2})\s*(?:Z\s*)?R\s*(\d+(?:\.\d+)?)', s, re.I)
    if m_cam2:
        w_in = float(m_cam2.group(1))
        width_mm = w_in * 25.4
        rim = float(m_cam2.group(2))
        return width_mm, np.nan, rim

    return np.nan, np.nan, np.nan

specs = df['Product Name'].apply(parse_tire_specs)
df['width'] = [s[0] for s in specs]
df['ratio'] = [s[1] for s in specs]
df['rim'] = [s[2] for s in specs]

print("Resumen de extracción de especificaciones:")
print(f"- Registros con ancho extraído: {df['width'].notna().sum()} / {len(df)}")
print(f"- Registros con rin extraído: {df['rim'].notna().sum()} / {len(df)}")
print(f"- Registros con ratio explícito o geométrico: {df['ratio'].notna().sum()} / {len(df)}")
print(f"- Registros con ratio faltante a imputar en Train/Test: {df['ratio'].isna().sum()} / {len(df)}")


## 4. División en Conjuntos de Entrenamiento (Train) y Prueba (Test)

Para evaluar la capacidad de generalización sobre datos no vistos, particionamos el dataset utilizando:
```python
train_test_split(X, y, test_size=0.20, random_state=42)
```
Con 251 observaciones, la partición efectiva produce:
* **Train:** **200 observaciones** ($79.68\%$)
* **Test:** **51 observaciones** ($20.32\%$)

La semilla `random_state=42` garantiza la exacta reproducibilidad de todos los resultados.


In [ ]:
X = df[['rim', 'width', 'ratio', 'Brand']].copy()
y = df['unit_cost_clean'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Dimensiones de X_train: {X_train.shape} (200 muestras)")
print(f"Dimensiones de X_test:  {X_test.shape} (51 muestras)")


## 5. Preprocesamiento: Imputación de Ratio y Codificación de Marca

Para respetar rigurosamente la separación de conjuntos y evitar cualquier **fuga de datos (data leakage)**:
1. **Imputación de Aspect Ratio:** Calculamos la mediana del ratio **únicamente sobre `X_train`** y utilizamos ese valor escalar para rellenar los faltantes tanto en `X_train` como en `X_test`.
2. **One-Hot Encoding de `Brand`:** Codificamos la marca utilizando variables *dummy*, estableciendo a **`BRAND_A`** como la categoría base de referencia (omitida). El conjunto `X_test` se alinea explícitamente con las columnas de `X_train` mediante reindexación (`reindex`).
3. **Escalado:** Se prescinde de `StandardScaler` para mantener los coeficientes expresados en magnitudes físicas naturales y en unidades monetarias directas (INR).


In [ ]:
# 1. Imputación de ratio calculada exclusivamente sobre Train
mediana_train_ratio = float(X_train['ratio'].dropna().median())
print(f"Mediana de Aspect Ratio en Train: {mediana_train_ratio:.2f}%")

X_train['ratio'] = X_train['ratio'].fillna(mediana_train_ratio)
X_test['ratio'] = X_test['ratio'].fillna(mediana_train_ratio)

# 2. One-Hot Encoding de Brand con base fija BRAND_A
X_train_enc = pd.get_dummies(X_train, columns=['Brand'], drop_first=False, dtype=float)
X_test_enc = pd.get_dummies(X_test, columns=['Brand'], drop_first=False, dtype=float)

# Alinear columnas de Test con Train
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0.0)

# Omitir BRAND_A como categoría de referencia
if 'Brand_BRAND_A' in X_train_enc.columns:
    X_train_enc = X_train_enc.drop(columns=['Brand_BRAND_A'])
    X_test_enc = X_test_enc.drop(columns=['Brand_BRAND_A'])

print("Variables predictoras finales del modelo múltiple:")
print(list(X_train_enc.columns))


## 6. Modelo 1 — Regresión Lineal Simple (Baseline)

Ajustamos un modelo de regresión lineal simple utilizando únicamente la variable que demostró mayor correlación lineal individual en el EDA:
$$\widehat{\text{Cost}} = \beta_0 + \beta_1 \cdot \text{Rim}$$


In [ ]:
# Entrenar modelo simple
modelo_simple = LinearRegression()
modelo_simple.fit(X_train_enc[['rim']], y_train)

print("Modelo simple entrenado exitosamente.")
print(f"Intercepto (beta_0): {modelo_simple.intercept_:.2f} INR")
print(f"Coeficiente Rim (beta_1): {modelo_simple.coef_[0]:.2f} INR por pulgada")


### Evaluación del Modelo Simple en Test
Generamos predicciones sobre las 51 observaciones no vistas del conjunto de prueba y calculamos las métricas formales:
* **MAE (Error Absoluto Medio):** promedio del valor absoluto de los errores en INR.
* **MSE (Error Cuadrático Medio):** promedio de los residuos al cuadrado.
* **RMSE (Raíz del Error Cuadrático Medio):** dispersión típica del error en INR, con mayor penalización a fallos grandes.
* **R² (Coeficiente de Determinación):** proporción de la varianza explicada respecto al promedio de la muestra.


In [ ]:
# Predicciones sobre Test
y_pred_simple = modelo_simple.predict(X_test_enc[['rim']])

mae_s = mean_absolute_error(y_test, y_pred_simple)
mse_s = mean_squared_error(y_test, y_pred_simple)
rmse_s = np.sqrt(mse_s)
r2_s = r2_score(y_test, y_pred_simple)

print("=== MÉTRICAS MODELO SIMPLE (TEST) ===")
print(f"MAE  : {mae_s:,.2f} INR")
print(f"MSE  : {mse_s:,.2f}")
print(f"RMSE : {rmse_s:,.2f} INR")
print(f"R²   : {r2_s:.4f}")


## 7. Modelo 2 — Regresión Lineal Múltiple (Integrador)

Incorporamos la geometría dimensional completa (`Rim`, `Width`, `Aspect Ratio`) y el impacto de cada fabricante (`Brand_BRAND_B`, `Brand_BRAND_C`, `Brand_BRAND_D`):
$$\widehat{\text{Cost}} = \beta_0 + \beta_1 \text{Rim} + \beta_2 \text{Width} + \beta_3 \text{Ratio} + \beta_4 B_B + \beta_5 B_C + \beta_6 B_D$$


In [ ]:
# Entrenar modelo múltiple
modelo_multiple = LinearRegression()
modelo_multiple.fit(X_train_enc, y_train)

print("Modelo múltiple entrenado exitosamente.")
df_coef = pd.DataFrame({
    'Variable': ['Intercepto (beta_0)'] + list(X_train_enc.columns),
    'Coeficiente': [modelo_multiple.intercept_] + list(modelo_multiple.coef_)
})
display(df_coef)


### Evaluación del Modelo Múltiple en Test
Calculamos las métricas de desempeño sobre las mismas 51 observaciones de prueba para contrastar la ganancia en precisión frente al baseline.


In [ ]:
# Predicciones sobre Test
y_pred_multiple = modelo_multiple.predict(X_test_enc)

mae_m = mean_absolute_error(y_test, y_pred_multiple)
mse_m = mean_squared_error(y_test, y_pred_multiple)
rmse_m = np.sqrt(mse_m)
r2_m = r2_score(y_test, y_pred_multiple)

print("=== MÉTRICAS MODELO MÚLTIPLE (TEST) ===")
print(f"MAE  : {mae_m:,.2f} INR")
print(f"MSE  : {mse_m:,.2f}")
print(f"RMSE : {rmse_m:,.2f} INR")
print(f"R²   : {r2_m:.4f}")


## 8. Comparación de Modelos y Diagnóstico de Generalización

Contrastamos el desempeño cuantitativo de ambos algoritmos:


In [ ]:
# Tabla comparativa de métricas en Test
tabla_resultados = pd.DataFrame({
    'Modelo': ['Regresión Lineal Simple (Rim)', 'Regresión Lineal Múltiple (Specs + Brand)'],
    'MAE (INR)': [round(mae_s, 2), round(mae_m, 2)],
    'MSE': [round(mse_s, 2), round(mse_m, 2)],
    'RMSE (INR)': [round(rmse_s, 2), round(rmse_m, 2)],
    'R²': [round(r2_s, 4), round(r2_m, 4)]
})
display(tabla_resultados)

# Diagnóstico Train vs Test para Modelo Múltiple
y_pred_train_m = modelo_multiple.predict(X_train_enc)
r2_train_m = r2_score(y_train, y_pred_train_m)
rmse_train_m = np.sqrt(mean_squared_error(y_train, y_pred_train_m))

print(f"\nDiagnóstico de Generalización (Modelo Múltiple):")
print(f"- R² Train: {r2_train_m:.4f} | R² Test: {r2_m:.4f}")
print(f"- RMSE Train: {rmse_train_m:,.2f} INR | RMSE Test: {rmse_m:,.2f} INR")


## 9. Visualizaciones Diagnósticas

Generamos tres figuras para el análisis técnico:
1. **Recta de Regresión Simple:** ajuste del baseline sobre los puntos de prueba.
2. **Valores Reales vs. Predichos:** dispersión comparativa frente a la recta identidad $y = x$.
3. **Gráfico de Residuos:** análisis de la dispersión de errores frente a las predicciones.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Gráfico 1: Recta Simple
axes[0].scatter(X_test_enc['rim'], y_test, color='#1f77b4', alpha=0.7, edgecolors='k', label='Datos reales (Test)')
x_line = np.linspace(X_test_enc['rim'].min(), X_test_enc['rim'].max(), 100)
y_line = modelo_simple.predict(pd.DataFrame({'rim': x_line}))
axes[0].plot(x_line, y_line, color='#d62728', linewidth=2, label='Recta ajustada')
axes[0].set_title('1. Regresión Simple (Rim vs Costo)', fontsize=11)
axes[0].set_xlabel('Diámetro del Rin (Pulgadas)')
axes[0].set_ylabel('Costo Unitario (INR)')
axes[0].legend()

# Gráfico 2: Real vs Predicho
axes[1].scatter(y_test, y_pred_simple, color='#ff7f0e', alpha=0.5, label='Simple')
axes[1].scatter(y_test, y_pred_multiple, color='#2ca02c', alpha=0.7, edgecolors='k', label='Múltiple')
min_v = min(y_test.min(), y_pred_multiple.min())
max_v = max(y_test.max(), y_pred_multiple.max())
axes[1].plot([min_v, max_v], [min_v, max_v], 'k--', linewidth=1.5, label='y = x')
axes[1].set_title('2. Valores Reales vs. Predichos (Test)', fontsize=11)
axes[1].set_xlabel('Costo Real (INR)')
axes[1].set_ylabel('Costo Predicho (INR)')
axes[1].legend()

# Gráfico 3: Residuos
residuos_m = y_test - y_pred_multiple
axes[2].scatter(y_pred_multiple, residuos_m, color='#9467bd', alpha=0.7, edgecolors='k')
axes[2].axhline(0, color='r', linestyle='--', linewidth=1.5)
axes[2].set_title('3. Residuos vs. Predicciones (Múltiple)', fontsize=11)
axes[2].set_xlabel('Costo Predicho (INR)')
axes[2].set_ylabel('Residuo (Real - Predicho) INR')

plt.tight_layout()
os.makedirs('../reports/figures', exist_ok=True)
plt.savefig('../reports/figures/diagnostico_regresion.png', dpi=300)
plt.show()


## 10. Conclusiones y Roadmap hacia la Evidencia N.º 3

### Análisis de Resultados:
1. **Capacidad del Baseline:** El diámetro del rin explica por sí solo aproximadamente el **54%** de la varianza del costo ($R^2 = 0.542$), confirmando la hipótesis del EDA. Sin embargo, su error medio ($\text{MAE} \approx 2.795$ INR) es elevado.
2. **Impacto del Modelo Múltiple:** Al incorporar el ancho de sección, la relación de aspecto y la marca, el coeficiente de determinación asciende notablemente ($R^2 \approx 0.80$) y el error absoluto medio se reduce a $\approx 1.678$ INR.
3. **Efecto de Fabricante:** Tomando a `BRAND_A` como base de referencia, `BRAND_C` exhibe un costo promedio significativamente menor ($\approx -1.308$ INR, marca económica), mientras que `BRAND_D` presenta una prima positiva sustancial ($\approx +4.649$ INR, productos de gama pesada).
4. **Generalización:** El desempeño entre entrenamiento y prueba se mantiene equilibrado sin evidencia de sobreajuste severo.

### Trabajo Futuro (Evidencia N.º 3):
De acuerdo con las pautas de la cátedra, las siguientes técnicas quedan reservadas para la Tercera Evidencia:
* **Transformación Logarítmica del Target:** Aplicación de $\ln(\text{Unit Cost})$ para corregir la asimetría residual visible en productos de costo extremo.
* **Regularización Lineal:** Modelos Ridge ($L_2$) y Lasso ($L_1$) para estabilizar coeficientes ante la correlación física entre ancho y rin.
* **Modelos de Ensamble:** Exploración de *Random Forest Regressor* para modelar patrones no lineales complejos.
